In [14]:
import numpy as np

arr = np.array([
    [1, 2],
    [3, 4],
    [1, 2],
    [5, 6]
])

targets = np.array([
    [1, 2],
    [5, 6]
])

mask = ~np.any((arr[:, None] == targets).all(axis=2), axis=1)
filtered = arr[mask]

print(filtered)

[[3 4]]


In [ ]:
import numpy as np
import numpy.typing as npt

manhatten_directions = np.array([
    [0, 1, 0, -1],
    [1, 0, -1, 0]
]).transpose()

cheby_shapehev_directions = np.array([
    [0, 1, 1, 1, 0, -1, -1, -1],
    [1, 1, 0, -1, -1, -1, 0, 1]
]).transpose()

directions_array = cheby_shapehev_directions

directions_array

def max_euclid(centroid, ucn):
    dists = np.linalg.norm(ucn - centroid, axis=1)
    max_dist = np.max(dists)
    return ucn[dists == max_dist], max_dist

def calculate_frontiers(used_nodes, drone_area, shape: tuple[int,int]):
    h, w = shape

    p_frontiers = drone_area[:, None, :] + directions_array[None, :, :] # generate all frontier candidates
    p_frontiers = p_frontiers.reshape(-1,2)

    mask = ( # out of bounds mask
        (p_frontiers[:, 0] >= 0) &
        (p_frontiers[:, 0] < h) &
        (p_frontiers[:, 1] >= 0) &
        (p_frontiers[:, 1] < w)
    )

    p_frontiers = p_frontiers[mask] # apply mask
    p_frontiers[~np.any((p_frontiers[:, None] == used_nodes).all(axis=2), axis=1)] # remove alreay_directions used nodes
    p_frontiers = np.unique(p_frontiers, axis=0) # remove duplicates
    return p_frontiers

def get_center(occupied: npt.NDArray):
    coords = np.argwhere(occupied)
    return np.mean(coords, axis=0)

def add_cell(i, x, y, drone_areas, drone_frontiers, occupied, h, w):
    drone_areas[i] = np.vstack([drone_areas[i], [x, y]])
    occupied[x, y] = True

    for x_directions, y_directions in directions_array:
        nx, ny = x + x_directions, y + y_directions

        if 0 <= nx < h and 0 <= ny < w and not occupied[nx, ny]:
            drone_frontiers[i].add((nx, ny))

def balanced_iterative_growth(shape: tuple[int, int], origins: list[tuple[int,int]]):
    h, w = shape

    occupied = np.zeros(shape, dtype=bool)



    n = len(origins)

    used_nodes = np.array(origins)
    drone_areas = [np.array([[x, y]]) for x, y in origins]
    
    total = h * w
    base = total // n
    remainder = total % n

    capacities = np.full(n, base)
    capacities[:remainder] += 1

    labels = np.full((h, w), -1, dtype=int)
    counts = np.full(n, 1, dtype=int)

    drone_frontiers = []

    drone_frontiers = [calculate_frontiers(used_nodes, drone_area, shape) for drone_area in drone_areas]

    # now use the shape (number of frontiers) to determine the drone that gets more area first

        

    flooding = True

In [32]:
origins = [
    (15,0),
    (16,0),
    (17,0),
    (18,0),
    (19,0),
]

shape = (20, 20)

labels = balanced_iterative_growth(shape, origins)

[[15  0]
 [16  0]
 [17  0]
 [18  0]
 [19  0]]
[[15  1]
 [16  1]
 [14  0]
 [14  1]]


(array([[14,  1]]), np.float64(3.1622776601683795))
[[16  1]
 [17  1]
 [15  1]]


(array([[15,  1]]), np.float64(2.23606797749979))
[[17  1]
 [18  1]
 [16  1]]


(array([[18,  1],
       [16,  1]]), np.float64(1.4142135623730951))
[[18  1]
 [19  1]
 [17  1]]


(array([[19,  1]]), np.float64(2.23606797749979))
[[19  1]
 [18  1]]


(array([[19,  1]]), np.float64(2.23606797749979))


In [10]:
import numpy as np
import numpy.typing as npt

# D_ANY[0] == [north_y, north_x]
# 4-directional movement (Manhatten distance)
D_MANHATTEN = np.array([
    [1, 0, -1, 0],# y
    [0, 1, 0, -1] # x
]).transpose()

# 8-directional movement (Chebyshev neighborhood)
D_CHEBYSHEV = np.array([
    [1, 1, 0, -1, -1, -1, 0, 1],# y
    [0, 1, 1, 1, 0, -1, -1, -1] # x
]).transpose()

class CMSG: # Constrained multi source growth
    def __init__(self, shape: tuple[int, int], origins: list[tuple[int, int]], directions: npt.NDArray):
        self.y_shape, self.x_shape = shape
        n = len(origins)
        self.directions = directions

        # --- Capacity ---
        total = self.y_shape * self.x_shape
        base = total // n
        remainder = total % n

        capacities = np.full(n, base)
        capacities[:remainder] += 1

        # --- State ---
        labels = -np.ones(shape, dtype=int)
        self.occupied = np.zeros(shape, dtype=bool)

        drone_areas = [[] for _ in range(n)]
        frontiers = [set() for _ in range(n)]
        counts = np.zeros(n, dtype=int)
        previous_count = np.zeros(n, dtype=int)
        
        # add starting nodes
        for i, (x_origin, y_origin) in enumerate(origins):
            i_origin = y_origin, x_origin
            labels[y_origin, x_origin] = i
            self.occupied[y_origin, x_origin] = True
            drone_areas[i].append(i_origin)
            counts[i] = 1

            for ny, nx in self.iter_neighbors(i_origin):
                if not self.occupied[ny, nx]:
                    frontiers[i].add((ny, nx))
        
        # --- Main loop ---
        while True:
            if np.array_equal(counts, previous_count):
                break

            previous_count = counts.copy()

            active = [i for i in range(n) if counts[i] < capacities[i]]
            if not active:
                break

            c_global = self.global_centroid() # compute global centroid once per iteration

            # iterate all drones (round robin) in order of ascending #frontiers
            for i in sorted(active, key=lambda d: len(frontiers[d])):

                if not frontiers[i]:
                    continue

                c_self = CMSG.centroid(drone_areas[i]) # drone area centroid

                best_score = -np.inf
                best_cell = None

                for frontier in frontiers[i]:
                    frontier_y, frontier_x = frontier
                    if self.occupied[frontier_y, frontier_x]:
                        continue

                    score = self.get_score(frontier, c_global, c_self)

                    if score > best_score:
                        best_score = score
                        best_cell = frontier

                if best_cell is None:
                    continue

                best_y, best_x = best_cell

                # remove from frontier
                frontiers[i].discard(best_cell)

                if self.occupied[best_y, best_x]:
                    continue

                # assign
                labels[best_y, best_x] = i
                self.occupied[best_y, best_x] = True
                drone_areas[i].append(best_cell)
                counts[i] += 1

                # expand frontier
                for ny, nx in self.iter_neighbors(best_cell):
                    if not self.occupied[ny, nx]:
                        frontiers[i].add((ny, nx))

        # if any unassigned cells remain assign greedy

        # --- Fill remaining unassigned cells by neighborhood majority ---
        unassigned = np.argwhere(labels == -1)

        while len(unassigned) > 0:
            progress = False

            for y, x in unassigned:
                neighbor_scores = np.zeros(n, dtype=float)

                for ny, nx in self.iter_neighbors((y, x)):
                    label = labels[ny, nx]
                    if label != -1:
                        # weighted contribution (favor smaller regions)
                        neighbor_scores[label] += 1.0 / (1 + counts[label])

                if np.any(neighbor_scores > 0):
                    # pick best score
                    max_score = np.max(neighbor_scores)
                    candidates = np.where(neighbor_scores == max_score)[0]

                    # tie-break: prefer smaller regions
                    i = min(candidates, key=lambda d: counts[d])

                    labels[y, x] = i
                    self.occupied[y, x] = True
                    counts[i] += 1

                    progress = True

            if not progress:
                # fallback: assign remaining cells to nearest centroid
                for y, x in unassigned:
                    i = np.argmin([
                        np.linalg.norm(np.array([y, x]) - CMSG.centroid(drone_areas[d]))
                        for d in range(n)
                    ])
                    labels[y, x] = i
                    counts[i] += 1
                break

            unassigned = np.argwhere(labels == -1)

        self.labels = labels
        self.counts = counts
    
    def iter_neighbors(self, coord: tuple[int,int]):
        y, x = coord
        for y_directions, x_directions in self.directions:
            ny, nx = y + y_directions, x + x_directions
            if 0 <= ny < self.y_shape and 0 <= nx < self.x_shape:
                yield ny, nx

    @staticmethod
    def centroid(area):
        pts = np.array(area)
        return np.mean(pts, axis=0)
    
    def global_centroid(self):
        coords = np.argwhere(self.occupied)
        return np.mean(coords, axis=0)
    
    def get_score(self, frontier: tuple[int,int], c_global: tuple[float,float], c_self: tuple[float,float]):
        p = np.array(frontier)

        # distances
        d_global = np.linalg.norm(p - c_global)
        d_self = np.linalg.norm(p - c_self)

        openness = 0
        for ny, nx in self.iter_neighbors(frontier):
            if not self.occupied[ny, nx]:
                openness += 1

        score = (
            2.0 * openness # main driver: go where space is
            - 0.5 * d_self # keep region coherent
            - 0.2 * d_global # mild global spreading pressure
        )

        occupied_neighbors = 0
        for ny, nx in self.iter_neighbors(frontier):
            if self.occupied[ny, nx]:
                occupied_neighbors += 1

        score -= 0.8 * occupied_neighbors

        return score
        

In [ ]:
origins = [
    (15,0),
    (16,0),
    (17,0),
    (18,0),
    (19,0),
]

# origins = [
#     (0,0),
#     (0,19),
#     (19,0),
#     (19,19),
#     (10,10),
# ]

shape = (20, 20)

cmsg = CMSG(shape, origins, D_MANHATTEN)
labels = cmsg.labels
counts = cmsg.counts

print(labels)
print(counts)

# todo create new battle function to redistribute nodes and steal

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 2 3 4]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 2 2 4 3]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 2 2 4 3 3]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 2 2 4 3 3]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 2 2 4 3 3]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 2 2 4 3 3]
 [0 0 0 0 0 0 0 1 0 1 0 1 1 1 1 2 2 4 3 3]
 [1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 2 2 4 3 3]
 [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 2 4 3 3]
 [1 1 1 1 1 1 1 1 1 2 2 2 1 2 1 2 2 4 3 3]
 [1 1 1 2 2 2 2 2 2 2 2 2 2 2 1 2 2 4 3 3]
 [1 1 1 2 2 2 2 2 2 2 2 2 2 2 1 2 2 4 3 3]
 [1 1 1 1 2 2 2 2 2 4 4 2 2 2 1 2 2 4 3 3]
 [1 1 1 1 2 2 2 2 2 4 4 4 2 2 1 2 2 4 3 3]
 [1 1 1 2 1 1 2 4 4 4 4 4 2 2 1 2 2 4 3 4]
 [1 1 1 1 1 1 2 4 4 4 4 4 2 2 1 2 2 4 3 4]
 [1 1 1 1 1 2 2 4 4 4 4 4 2 4 1 2 2 4 3 4]
 [1 4 4 4 4 4 4 4 4 4 4 4 2 4 1 2 2 4 3 4]
 [4 4 4 4 4 4 4 4 4 4 4 4 4 2 2 2 2 4 3 4]
 [4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 4 3 4 3]]
[96 92 91 33 88]
